In [1]:
import sys
import os
import gc
from pathlib import Path

import pandas as pd

sys.path.append(os.path.abspath("../../../"))
PROJECT_ROOT = "../../../"

from preprocessing.EEGMotorMovement.preprocessing import (
    load_physionet_eegmmidb_data
)
from preprocessing.general.filtering import (
    apply_filters_to_dataset,
    bands
)
import preprocessing.general.feature_extraction as fe


# ============================================================
# Dataset metadata
# ============================================================

DATASET_NAME = "eegmmidb"

# EEGMMIDB does not define separate acquisition sessions.
SESSION_NAME = "session_01"


# ============================================================
# Channel configuration
# ============================================================

# Simple identifier for this electrode configuration.
ELECTRODE_SETUP = "setup_01"


# All electrodes shared with BCI Competition IV 2a.
"""
selected_channels = [
    "Fz",
    "FC3",
    "FC1",
    "FCz",
    "FC2",
    "FC4",
    "C5",
    "C3",
    "C1",
    "Cz",
    "C2",
    "C4",
    "C6",
    "CP3",
    "CP1",
    "CPz",
    "CP2",
    "CP4",
    "P1",
    "Pz",
    "P2",
    "POz",
]
"""

# Electrode configuration used by setup_01.
selected_channels = [
    "C3",
    "Cz",
    "C4",
]

# Use all available PhysioNet EEG channels instead:
# selected_channels = None


# ============================================================
# Paths
# ============================================================

root_edf = os.path.join(
    PROJECT_ROOT,
    "Datasets/EEG Motor Movement/original/files/"
)

output_csv = Path(
    os.path.join(
        PROJECT_ROOT,
        "Datasets/EEG Motor Movement/processed/",
        f"EEG_MM_features_{ELECTRODE_SETUP}.csv",
    )
)


# ============================================================
# Feature configuration
# ============================================================

extract_config = {
    "mean": {"function": fe.extract_mean},
    "std": {"function": fe.extract_std},
    "mom": {"function": fe.extract_moments},
    "min": {"function": fe.extract_min},
    "max": {"function": fe.extract_max},
    "cov": {"function": fe.extract_covariance},
    "eig": {"function": fe.extract_eigenvalues},

    # "logcov": {
    #     "function": fe.extract_logcov
    # },

    # "fft": {
    #     "function": fe.extract_fft,
    #     "params": {"ntop": 5},
    # },

    "h_diff": {"function": fe.extract_halves_diff},
    "q_stats": {"function": fe.extract_quarters_stats},
    "logvar": {"function": fe.extract_logvar},
}


# ============================================================
# Incremental processing configuration
# ============================================================

subjects = list(range(1, 110))

# Reduce to 1 for minimum memory usage.
SUBJECT_BATCH_SIZE = 5


# ============================================================
# Prepare output
# ============================================================

output_csv.parent.mkdir(
    parents=True,
    exist_ok=True,
)

# Prevent results from being appended to an old file.
if output_csv.exists():
    output_csv.unlink()

first_write = True
total_rows = 0


# ============================================================
# Load, filter, extract, validate, and save incrementally
# ============================================================

for start in range(
    0,
    len(subjects),
    SUBJECT_BATCH_SIZE,
):

    subject_batch = subjects[
        start:start + SUBJECT_BATCH_SIZE
    ]

    print(
        f"\nProcessing subjects "
        f"{subject_batch[0]}–{subject_batch[-1]}"
    )

    # --------------------------------------------------------
    # Load current batch
    # --------------------------------------------------------

    batch_data = load_physionet_eegmmidb_data(
        root_dir=root_edf,
        config={
            "subjects": subject_batch,
            "channels": selected_channels,
        },
    )

    if not batch_data:
        print("⚠️ No data loaded for this batch.")
        continue

    print("✅ Data loading complete.")

    # --------------------------------------------------------
    # Filtering and resampling
    # --------------------------------------------------------

    filtered_data = apply_filters_to_dataset(
        dataset=batch_data,
        config={
            "original_fs": 160,
        },
    )

    print("✅ Filtering complete.")

    # --------------------------------------------------------
    # Feature extraction
    # --------------------------------------------------------

    df_batch = fe.extract_features_to_dataframe(
        dataset=filtered_data,
        extract_config=extract_config,
        band_labels=bands,
        dataset_name=DATASET_NAME,
        session_name=SESSION_NAME,
    )

    if df_batch.empty:
        print("⚠️ No features generated for this batch.")

        del batch_data
        del filtered_data
        del df_batch

        gc.collect()
        continue

    print(
        f"✅ Feature extraction complete: "
        f"{df_batch.shape}"
    )

    # --------------------------------------------------------
    # Validate current batch
    # --------------------------------------------------------

    fe.validate_feature_dataframe(df_batch)

    print("✅ Batch validation complete.")

    # --------------------------------------------------------
    # Append batch to CSV
    # --------------------------------------------------------

    df_batch.to_csv(
        output_csv,
        mode="w" if first_write else "a",
        header=first_write,
        index=False,
    )

    if first_write:
        display(df_batch.head())
        first_write = False

    total_rows += len(df_batch)

    print(
        f"✅ Batch saved. "
        f"Total rows written: {total_rows}"
    )

    # --------------------------------------------------------
    # Release memory
    # --------------------------------------------------------

    del batch_data
    del filtered_data
    del df_batch

    gc.collect()


# ============================================================
# Final validation and report
# ============================================================

if first_write:
    print("⚠️ No feature data were written.")

else:
    print("\nValidating complete saved dataset...")

    df_complete = pd.read_csv(output_csv)

    fe.validate_feature_dataframe(df_complete)

    if len(df_complete) != total_rows:
        raise ValueError(
            "The number of rows in the saved CSV does not match "
            f"the number written: {len(df_complete)} versus "
            f"{total_rows}."
        )

    print("✅ Complete dataset validation passed.")
    print("✅ Complete feature extraction finished.")
    print(f"✅ Electrode setup: {ELECTRODE_SETUP}")
    print(f"✅ Channels: {selected_channels}")
    print(f"✅ Total rows: {total_rows}")
    print(f"✅ Saved to: {output_csv}")

    del df_complete
    gc.collect()

/Users/edsonodake/miniforge3/envs/svm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Processing subjects 1–5


Loading PhysioNet EEGMMIDB: 100%|██████████| 5/5 [00:02<00:00,  2.46subject/s]


✅ Data loading complete.
✅ Filtering complete.


Subjects: 100%|██████████| 5/5 [00:01<00:00,  3.92it/s]

...

✅ Feature extraction complete: (897, 95)
✅ Batch validation complete.
✅ Batch saved. Total rows written: 18964

Processing subjects 106–109


Loading PhysioNet EEGMMIDB: 100%|██████████| 4/4 [00:01<00:00,  3.05subject/s]


✅ Data loading complete.
✅ Filtering complete.


Subjects: 100%|██████████| 4/4 [00:00<00:00,  4.17it/s]


✅ Feature extraction complete: (709, 95)
✅ Batch validation complete.
✅ Batch saved. Total rows written: 19673

Validating complete saved dataset...
✅ Complete dataset validation passed.
✅ Complete feature extraction finished.
✅ Electrode setup: setup_01
✅ Channels: ['C3', 'Cz', 'C4']
✅ Total rows: 19673
✅ Saved to: ../../../Datasets/EEG Motor Movement/processed/EEG_MM_features_setup_01.csv
